# ANN Churn Prediction Model with Cross-Validation, Grid Search, and Checkpointing

In [ ]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.wrappers.scikit_learn import KerasClassifier
from tensorflow.keras.callbacks import ModelCheckpoint


In [ ]:

# Load dataset
df = pd.read_csv("churn_data.csv")

X = df.drop("churn", axis=1)
y = df["churn"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:

def build_model(optimizer='adam', neurons=32, dropout_rate=0.2):
    model = Sequential()

    model.add(Dense(neurons, input_dim=X_train.shape[1], activation='relu'))
    model.add(Dropout(dropout_rate))

    model.add(Dense(neurons//2, activation='relu'))
    model.add(Dropout(dropout_rate))

    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model


In [ ]:

model = KerasClassifier(build_fn=build_model, verbose=0)

param_grid = {
    'batch_size': [16, 32],
    'epochs': [20, 50],
    'optimizer': ['adam', 'rmsprop'],
    'neurons': [32, 64],
    'dropout_rate': [0.2, 0.3]
}

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=kfold,
    n_jobs=-1
)

grid_result = grid.fit(X_train, y_train)

print("Best Score:", grid_result.best_score_)
print("Best Params:", grid_result.best_params_)


In [ ]:

best_params = grid_result.best_params_

final_model = build_model(
    optimizer=best_params['optimizer'],
    neurons=best_params['neurons'],
    dropout_rate=best_params['dropout_rate']
)

checkpoint = ModelCheckpoint(
    "best_model.h5",
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

history = final_model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=best_params['epochs'],
    batch_size=best_params['batch_size'],
    callbacks=[checkpoint],
    verbose=1
)


In [ ]:

loss, accuracy = final_model.evaluate(X_test, y_test)
print("Test Accuracy:", accuracy)
